## Hybrid AML Detection Model
This model combines the Isolation Forest with Rule-Based Scoring.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [3]:
BASE_DIR = '/content/gdrive/MyDrive/AML_Competition'
MODEL_DIR = Path(BASE_DIR) / 'models'

### Load Scores From Both Models

In [4]:
iso_results = pd.read_csv(MODEL_DIR / 'isolation_forest_results.csv')
print(f"   Isolation Forest: {len(iso_results):,} customers")

rule_results = pd.read_csv(MODEL_DIR / 'rule_based_scores.csv')
print(f"   Rule-Based System: {len(rule_results):,} customers")

   Isolation Forest: 61,410 customers
   Rule-Based System: 61,410 customers


### Merge and Create Hybrid Scores

In [5]:
hybrid_df = iso_results[['customer_id', 'risk_score', 'prediction', 'actual_label']].merge(
    rule_results[['customer_id', 'rule_based_score', 'rules_triggered', 'evidence']],
    on='customer_id',
    how='inner'
)

print(f"   Merged {len(hybrid_df):,} customers")

# Test different weighting strategies
weighting_strategies = {
    'ML_Heavy': {'iso': 0.70, 'rule': 0.30},
    'Balanced': {'iso': 0.50, 'rule': 0.50},
    'Rule_Heavy': {'iso': 0.30, 'rule': 0.70},
    'Adaptive': None
}

best_strategy = None
best_performance = 0
results_summary = []

for strategy_name, weights in weighting_strategies.items():
    if strategy_name == 'Adaptive':
        # Adaptive: Use rule score more heavily when many rules triggered
        hybrid_df['adaptive_rule_weight'] = np.minimum(
            0.3 + (hybrid_df['rules_triggered'] * 0.1), 0.7
        )
        hybrid_df[f'hybrid_score_{strategy_name}'] = (
            hybrid_df['risk_score'] * (1 - hybrid_df['adaptive_rule_weight']) +
            hybrid_df['rule_based_score'] * hybrid_df['adaptive_rule_weight']
        )
    else:
        # Fixed weights
        hybrid_df[f'hybrid_score_{strategy_name}'] = (
            hybrid_df['risk_score'] * weights['iso'] +
            hybrid_df['rule_based_score'] * weights['rule']
        )

    # Calculate performance if labels available
    labeled_mask = hybrid_df['actual_label'].notna()

    if labeled_mask.sum() > 0:
        score_col = f'hybrid_score_{strategy_name}'

        # Check if we have any suspicious customers
        susp_indices = set(hybrid_df[hybrid_df['actual_label'] == 1].index)

        if len(susp_indices) == 0:
            print(f"      {strategy_name:15s}: No suspicious customers - skipping")
            continue

        # ROC-AUC
        try:
            roc_auc = roc_auc_score(
                hybrid_df.loc[labeled_mask, 'actual_label'],
                hybrid_df.loc[labeled_mask, score_col]
            )
        except:
            roc_auc = 0.0

        # Capture rates - use FULL dataset percentiles, not just labeled
        total_customers = len(hybrid_df)
        top_1pct_count = max(1, int(total_customers * 0.01))   # Top 1% of ALL customers
        top_5pct_count = max(1, int(total_customers * 0.05))   # Top 5% of ALL customers
        top_10pct_count = max(1, int(total_customers * 0.10))  # Top 10% of ALL customers

        # Get indices of top customers from FULL dataset
        sorted_all = hybrid_df.sort_values(score_col, ascending=False).index

        top_1pct_indices = set(sorted_all[:top_1pct_count])
        top_5pct_indices = set(sorted_all[:top_5pct_count])
        top_10pct_indices = set(sorted_all[:top_10pct_count])

        # Count how many suspicious customers are in each top percentile
        capture_1pct = len(susp_indices & top_1pct_indices) / len(susp_indices)
        capture_5pct = len(susp_indices & top_5pct_indices) / len(susp_indices)
        capture_10pct = len(susp_indices & top_10pct_indices) / len(susp_indices)

        # Track performance
        perf_score = capture_5pct  # Use 5% capture rate as primary metric

        results_summary.append({
            'strategy': strategy_name,
            'weights': f"{weights['iso']:.0%}/{weights['rule']:.0%}" if weights else "Adaptive",
            'roc_auc': roc_auc,
            'capture_1pct': capture_1pct,
            'capture_5pct': capture_5pct,
            'capture_10pct': capture_10pct
        })

        if perf_score > best_performance:
            best_performance = perf_score
            best_strategy = strategy_name

        print(f"      {strategy_name:15s}: ROC-AUC={roc_auc:.3f}, "
              f"Capture@5%={capture_5pct*100:.1f}%")

   Merged 61,410 customers
      ML_Heavy       : ROC-AUC=0.831, Capture@5%=10.0%
      Balanced       : ROC-AUC=0.844, Capture@5%=30.0%
      Rule_Heavy     : ROC-AUC=0.799, Capture@5%=20.0%
      Adaptive       : ROC-AUC=0.820, Capture@5%=0.0%


### Select Best Strategy


In [8]:
results_summary_df = pd.DataFrame(results_summary)
print(f"\n Strategy Comparison:")
print(results_summary_df.to_string(index=False))

# Use best strategy as final hybrid score
hybrid_df['final_hybrid_score'] = hybrid_df[f'hybrid_score_{best_strategy}']

hybrid_df = hybrid_df.sort_values('final_hybrid_score', ascending=False)

# Add risk category
def categorize_risk(score):
    if score >= 0.80: return 'Very High'
    elif score >= 0.60: return 'High'
    elif score >= 0.40: return 'Medium'
    elif score >= 0.20: return 'Low'
    else: return 'Very Low'

hybrid_df['hybrid_risk_category'] = hybrid_df['final_hybrid_score'].apply(categorize_risk)


 Strategy Comparison:
  strategy  weights  roc_auc  capture_1pct  capture_5pct  capture_10pct
  ML_Heavy  70%/30% 0.831111           0.0           0.1            0.3
  Balanced  50%/50% 0.844040           0.0           0.3            0.3
Rule_Heavy  30%/70% 0.799192           0.0           0.2            0.3
  Adaptive Adaptive 0.820202           0.0           0.0            0.3


### Final Results Summary

In [10]:
print(f"\n{'='*70}")
print(f"FINAL MODEL: {best_strategy} Strategy")
print(f"{'='*70}")

print(f"\n Performance Metrics:")

labeled_mask = hybrid_df['actual_label'].notna()

if labeled_mask.sum() > 0 and len(results_summary) > 0:
    # Get the best result
    best_result = results_summary_df[results_summary_df['strategy'] == best_strategy].iloc[0]

    print(f"   ROC-AUC: {best_result['roc_auc']:.3f}")
    print(f"   Capture@1%:  {best_result['capture_1pct']*100:.1f}%")
    print(f"   Capture@5%:  {best_result['capture_5pct']*100:.1f}%")
    print(f"   Capture@10%: {best_result['capture_10pct']*100:.1f}%")

    print(f"\n Comparison to Baseline Models:")

    # Calculate baseline performances
    iso_roc = roc_auc_score(
        hybrid_df.loc[labeled_mask, 'actual_label'],
        hybrid_df.loc[labeled_mask, 'risk_score']
    )

    rule_roc = roc_auc_score(
        hybrid_df.loc[labeled_mask, 'actual_label'],
        hybrid_df.loc[labeled_mask, 'rule_based_score']
    )

    print(f"   Isolation Forest alone:  ROC-AUC = {iso_roc:.3f}")
    print(f"   Rule-Based alone:        ROC-AUC = {rule_roc:.3f}")
    print(f"   {best_strategy} Hybrid:  ROC-AUC = {best_result['roc_auc']:.3f}")
    print(f"   Improvement:             +{(best_result['roc_auc'] - iso_roc)*100:.1f}% over Isolation Forest")

    print(f"\n Why {best_strategy} is Best:")
    print(f"   - Combines ML pattern detection (Isolation Forest)")
    print(f"   - With AML domain expertise (Rule-Based)")
    print(f"   - Achieves highest Capture@5%: {best_result['capture_5pct']*100:.0f}%")
    print(f"   - Meaning: Top 5% of flagged customers contain {best_result['capture_5pct']*100:.0f}% of suspicious activity")

print(f"\n{'='*70}")


FINAL MODEL: Balanced Strategy

 Performance Metrics:
   ROC-AUC: 0.844
   Capture@1%:  0.0%
   Capture@5%:  30.0%
   Capture@10%: 30.0%

 Comparison to Baseline Models:
   Isolation Forest alone:  ROC-AUC = 0.794
   Rule-Based alone:        ROC-AUC = 0.758
   Balanced Hybrid:  ROC-AUC = 0.844
   Improvement:             +5.0% over Isolation Forest

 Why Balanced is Best:
   - Combines ML pattern detection (Isolation Forest)
   - With AML domain expertise (Rule-Based)
   - Achieves highest Capture@5%: 30%
   - Meaning: Top 5% of flagged customers contain 30% of suspicious activity



### Save Results

In [15]:
# Save full results
output_path = MODEL_DIR / 'hybrid_model_results.csv'
hybrid_df.to_csv(output_path, index=False)
print(f"   Full results: {output_path}")

# Save high-risk only
high_risk_path = MODEL_DIR / 'hybrid_model_high_risk.csv'
high_risk = hybrid_df[hybrid_df['final_hybrid_score'] > 0.60]
high_risk.to_csv(high_risk_path, index=False)
print(f"   High-risk subset: {high_risk_path} ({len(high_risk):,} customers)")

# Save performance summary
summary_path = MODEL_DIR / 'hybrid_model_summary.csv'
results_summary_df.to_csv(summary_path, index=False)
print(f"   Performance summary: {summary_path}")

submission = hybrid_df[['customer_id', 'final_hybrid_score']].copy().reset_index(drop=True)

# Add predicted_label column (1 = flagged for investigation, 0 = not flagged)
# Flag top 1% as high-risk for investigation
threshold_1pct = submission['final_hybrid_score'].quantile(0.99)
submission['predicted_label'] = (submission['final_hybrid_score'] >= threshold_1pct).astype(int)

# Rename and reorder columns to match competition format
submission = submission[['customer_id', 'predicted_label', 'final_hybrid_score']]
submission.columns = ['customer_id', 'predicted_label', 'risk_score']

# Verify all customers are included
print(f"   Total customers: {len(submission):,}")
print(f"   Flagged for investigation: {submission['predicted_label'].sum():,} (top 1%)")
print(f"   Risk score range: [{submission['risk_score'].min():.3f}, {submission['risk_score'].max():.3f}]")


   Full results: /content/gdrive/MyDrive/AML_Competition/models/hybrid_model_results.csv
   High-risk subset: /content/gdrive/MyDrive/AML_Competition/models/hybrid_model_high_risk.csv (58 customers)
   Performance summary: /content/gdrive/MyDrive/AML_Competition/models/hybrid_model_summary.csv
   Total customers: 61,410
   Flagged for investigation: 615 (top 1%)
   Risk score range: [0.042, 0.756]
